In [9]:
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
drive.mount("/content/drive")
data = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/housing.csv")

data.info()

Mounted at /content/drive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [3]:
data_norm = pd.DataFrame(data)
data_norm['area'] = (data_norm['area'] - data_norm['area'].min()) / (data_norm['area'].max() - data_norm['area'].min())
data_norm['price'] = (data_norm['price'] - data_norm['price'].min()) / (data_norm['price'].max() - data_norm['price'].min())
data_norm['bedrooms'] = (data_norm['bedrooms'] - data_norm['bedrooms'].min()) / (data_norm['bedrooms'].max() - data_norm['bedrooms'].min())
data_norm['bathrooms'] = (data_norm['bathrooms'] - data_norm['bathrooms'].min()) / (data_norm['bathrooms'].max() - data_norm['bathrooms'].min())
data_norm.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,1.000000,0.396564,0.6,0.333333,3,yes,no,no,no,yes,2,yes,furnished
1,0.909091,0.502405,0.6,1.000000,4,yes,no,no,no,yes,3,no,furnished
2,0.909091,0.571134,0.4,0.333333,2,yes,no,yes,no,no,2,yes,semi-furnished
3,0.906061,0.402062,0.6,0.333333,2,yes,no,yes,no,yes,3,yes,furnished
4,0.836364,0.396564,0.6,0.000000,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
data_train = pd.DataFrame(data_norm[:451])
data_test = pd.DataFrame(data_norm[451:])

In [26]:
def regression(w0, w1, w2, w3, df: pd.DataFrame):
  res = []
  for row in df.itertuples():
    y = w0 + w1*row.area + w2*row.bedrooms + w3*row.bathrooms
    res.append(y)
  return pd.Series(res)

def err_foo(df: pd.DataFrame):
  res = []
  for row in df.itertuples():
    e = (row.y - row.h)**2
    res.append(e)
  res = pd.Series(res)
  j = res.sum()/(2*len(res))
  return j

def err_mean(df: pd.DataFrame):
  res = []
  for row in df.itertuples():
    e = abs(row.h - row.y)
    res.append(e)
  res = pd.Series(res)
  j = res.sum()/(len(res))
  return j

def iteration(w0, w1, w2, w3, df: pd.DataFrame):
  a = 0.001
  w0n = w0 - a*(df['e'].mean())
  w1n = w1 - a*(df['ex1'].mean())
  w2n = w2 - a*(df['ex2'].mean())
  w3n = w3 - a*(df['ex3'].mean())
  return (w0n, w1n, w2n, w3n)

w0 = 4
w1 = 4
w2 = 4
w3 = 4
e = 50
count = 0

# В завданні не сказано, скільки циклів спуску робити, тому я вірішив йти від середньоквадратичної помилки
while(e > 0.01):

  if count % 100 == 0:
    print(f"\nW0: {w0}\nW2: {w1}\nW3: {w2}\nW4: {w3}")
    print(f"\nПомилка: {e}")
  h = regression(w0, w1, w2, w3, data_norm)
  df = pd.DataFrame({
    "h": h,
    "y": data_norm['price'],
    "e": h - data_norm['price'],
    "ex1": (h - data_norm['price'])*data['area'],
    "ex2": (h - data_norm['price'])*data['bedrooms'],
    "ex3": (h - data_norm['price'])*data['bathrooms'],
    })

  e = err_foo(df)
  w0, w1, w2, w3 = iteration(w0, w1, w2, w3, df)
  count += 1


print(f"\nW0: {w0}\nW1: {w1}\nW2: {w2}\nW3: {w3}")
print(f"\nПомилка: {e}")


W0: 4
W2: 4
W3: 4
W4: 4

Помилка: 50

W0: 3.8859662453396644
W2: -18.042900938964273
W3: 3.6373001540438175
W4: 3.840649545146338

Помилка: 4.287546758011444

W0: 3.7803607204940337
W2: -17.19768373636141
W3: 3.302820530116863
W4: 3.6933152965330467

Помилка: 3.900347075510686

W0: 3.6794201922995367
W2: -16.39245915318814
W3: 2.9850070226334413
W4: 3.5528964334130464

Помилка: 3.549183268069269

W0: 3.5829246640255596
W2: -15.625328133054627
W3: 2.6830643862553654
W4: 3.4190648421338197

Помилка: 3.2307008521009952

W0: 3.4906645821678435
W2: -14.894481827563835
W3: 2.396235168573514
W4: 3.291507993663107

Помилка: 2.9418571550188

W0: 3.4024403403950165
W2: -14.198197311156662
W3: 2.12379791474811
W4: 3.169928203278033

Помилка: 2.6798923589131745

W0: 3.318061807059093
W2: -13.534833499515535
W3: 1.8650654574346546
W4: 3.0540419254213695

Помилка: 2.4423032317907074

W0: 3.2373478751506295
W2: -12.902827261857055
W3: 1.619383287944241
W4: 2.9435790820543155

Помилка: 2.226819297005

In [10]:
X = pd.DataFrame({
    "area": data_norm['area'],
    "bedrooms": data_norm['bedrooms'],
    "bathrooms": data_norm['bathrooms']
})
y = pd.Series(data_norm['price'])
regressor = LinearRegression().fit(X, y)
print(f"W0: {regressor.intercept_}\n W1-W3: {regressor.coef_}")
y_pred_train = regressor.predict(X)
mse_train = mean_squared_error(y, y_pred_train)
print(f"MSE: {mse_train}")

W0: 0.04282739976995409
 W1-W3: [0.47714269 0.17611257 0.36001286]
MSE: 0.01342681021702981


In [ ]:
#plt.figure(figsize=(14, 6))
#plt.scatter(df['y'], df['h'], color='blue')
#plt.title("Розбіжність прогнозованих та реальних даних", color='green', fontsize = 16)
#plt.ylabel("Прогнозовані дані", color='green', fontsize=12)
#plt.xlabel("Реальні дані", color='green', fontsize=12)
#plt.show()